### RAG Pipeline - Data INgestion to VEctor DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\abhin\AppData\Local\Temp\ipykernel_16140\2821909192.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader
c:\Users\abhin\Dropbox\PC\Downloads\projects\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory."""

    all_documents = []
    pdf_dir = Path(pdf_directory)
    for pdf_file in pdf_dir.glob("**/*.pdf"):
        print(f"Processing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = str(pdf_file)
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Processed {len(documents)} documents from {pdf_file}.")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    print(f"Total documents processed: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/PDF")

Processing ..\data\PDF\Abhinav_Singh_Resume.pdf...
Processed 1 documents from ..\data\PDF\Abhinav_Singh_Resume.pdf.
Processing ..\data\PDF\ITR2-Preview.pdf...
Processed 47 documents from ..\data\PDF\ITR2-Preview.pdf.
Processing ..\data\PDF\Profile.pdf...
Processed 3 documents from ..\data\PDF\Profile.pdf.
Total documents processed: 51


In [3]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"First chunk: {split_docs[0].page_content[:100]}...")
        print(f"Metadata of first chunk: {split_docs[0].metadata}")
        print(f"Last chunk: {split_docs[-1].page_content[:100]}...")

    return split_docs


In [4]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Splitted 51 documents into 146 chunks.
First chunk: Abhinav Singh
 leetcode.com/abhinavsingh
ï linkedin.com/in/abhinav-singh
# abhinavsingh649@gmail.co...
Metadata of first chunk: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-06T13:58:22+00:00', 'source': '..\\data\\PDF\\Abhinav_Singh_Resume.pdf', 'file_path': '..\\data\\PDF\\Abhinav_Singh_Resume.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-07-06T13:58:22+00:00', 'trapped': '', 'modDate': 'D:20260706135822Z', 'creationDate': 'D:20260706135822Z', 'page': 0, 'source_file': '..\\data\\PDF\\Abhinav_Singh_Resume.pdf', 'file_type': 'pdf'}
Last chunk: Data Science and Machine Learning, Data Processing · (January
2025 - January 2026)
Indian Institute ...


In [5]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'pandas'